# Notebook for the Winter cours on Algorithms for Structural Bioinformatics

https://algosb2023.loria.fr/

This notebook is for the practical session on "The image alignment problem" by Carlos Oscar Sorzano. Special thanks to **Jelena Banjac** (https://jelenabanjac.com/protein-reconstruction/README.html) for the inspiration to prepare the practicals as a Jupyter Notebook, and **Oier Lauzirika** for solving installation issues with the notebook.

# Installation

We will first install all the dependencies needed for the execution of our notebook.

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
%%writefile algosb_alignment.yml

name: algosb_alignment

channels:
  - astra-toolbox
  - conda-forge
  - anaconda
  - defaults

dependencies:
  - numpy
  - ipywidgets=8.0.2
  - ipyvolume
  - h5py
  - seaborn
  - astra-toolbox
  - configargparse
  - plotly
  - scikit-learn
  - scikit-image
  - nglview
  - mrcfile
  - tensorflow

  - pip
  - pip:
    - tensorflow-graphics
#    - matplotlib==3.1.3


In [ ]:
%%capture output
!mamba env update -n base -f algosb_alignment.yml

In [ ]:
# https://jelenabanjac.com/protein-reconstruction/README.html
%%capture output
!git clone https://github.com/oierlauzi/protein-reconstruction.git

# Data generation

Now we start with the data generation

In [ ]:
import nglview as nv
import h5py
import numpy as np
import scipy
import scipy.ndimage as ndimage
import matplotlib.pyplot as plt
import pandas as pd
import math

import sys
sys.path.append('protein-reconstruction')
import cryoem

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

Support for third party widgets will remain active for the duration of the session. To disable support:

In [ ]:
from google.colab import output
output.disable_custom_widget_manager()

If the viewer does not open, you may see it at https://www.rcsb.org/3d-view/6IDF

In [ ]:
view_atomic = nv.show_pdbid("6idf")
view_atomic

# Generation of the map

In ChimeraX, we will open the PDB and convert it to a map whose atoms will have 2A:

```
open 6idf
measure center #1
vop new #2 gridSpacing 0.75 size 250,250,250 origin 87,88,94
molmap #1 2 onGrid #2
save C:/Users/yourUser/Desktop/6idf.mrc models #3
```

In [ ]:
pdbID="6idf"

In [ ]:
#Alternatively you may download it from
!wget -O {pdbID}.mrc --no-check-certificate 'https://labur.eus/VbBGF' #https://docs.google.com/uc?export=download&id=1q2QkvjaUAmmOr6gb4uyCNbZxzE9ZWn9s'

# Generation of the projections

We will use the projection generator from Jelena Banjac

In [ ]:
!python protein-reconstruction/generator.py --help

In [ ]:
aux="""
input-file: %s.mrc
output-file: projections.h5

[projections]
projections-num: 500

[angles]
angle-shift: [0, 0, 0]
angle-coverage: [6.28, 3.14, 6.28]
"""%pdbID
with open("projection.config", "w") as text_file:
    text_file.write(aux)

!python protein-reconstruction/generator.py --config-file projection.config -ang-gen uniform_angles

The size of the projections is 433=250$\sqrt{3}$ to allow the full cube to fit in the projection.

![CubeDiagonal](https://drive.google.com/uc?export=view&id=19jD3h9fDOTO_LYA4y08P-0VvMNqNcu20 "blog-image align")

In [ ]:
projections_filename = "projections.h5"

# load projections
projectionsH5 = h5py.File(projections_filename, 'r')
angles = np.array(projectionsH5['Angles'], dtype=np.float32)
projections = np.array(projectionsH5['Projections'], dtype=np.float32)

print(f"{angles.shape[0]} projections of images with dimension {projections.shape[1:]} pixels")
print(f"{angles.shape[0]} sets of {angles.shape[1]} ground truth angles of corresponding projection images")

In [ ]:
def crop_to_size(projectionStack,newSizeX,newSizeY):
    n,y,x = projectionStack.shape
    startx = x//2 - newSizeX//2
    starty = y//2 - newSizeY//2
    return projectionStack[:, starty:starty+newSizeY, startx:startx+newSizeX]
projections = crop_to_size(projections,250,250)

In [ ]:
from cryoem.plots import plot_angles_histogram
plot_angles_histogram([angles], plot_settings=dict(figsize=(10, 4)))

In [ ]:
from cryoem.plots import plot_projections
pids = range(10)
plot_projections(projections[pids], [f'Projection {pid}\nAngles {list(map(lambda x: round(x,2) , angles[pid]))}' for pid in pids], nrows=2, ncols=5)

In [ ]:
from cryoem.plots import plot_detector_pixels_with_protein
plot_detector_pixels_with_protein(angles, "%s.mrc"%pdbID)


In [ ]:
from cryoem.plots import plot_images
plot_images(angles, projections, indices=range(50), img_size_scale=0.2)

In [ ]:
from cryoem.plots import plot_rays
plot_rays(angles, indices=range(50))

# Useful functions

In [ ]:
import astra
from cryoem.rotation_matrices import RotationMatrix
def generate_projections_ASTRA(vol, angles):
    """Generate projections with ASTRA toolbox

    """
    projSize=vol.shape[0]
    angles=np.reshape(np.deg2rad(angles),(1,3))
    vol_geom    = astra.create_vol_geom(vol.shape[1], vol.shape[2], vol.shape[0])
    orientation_Vectors   = RotationMatrix(angles)
    proj_geom = astra.create_proj_geom('parallel3d_vec', projSize, projSize, orientation_Vectors)
    _, proj_data = astra.create_sino3d_gpu(vol, proj_geom, vol_geom)
    projection = np.transpose(proj_data, (1, 0, 2))
    return np.reshape(projection,(projSize,projSize))

In [ ]:
def MSE(img1, img2):
    return np.linalg.norm(img1-img2,2)

def MAE(img1, img2):
    return np.linalg.norm(img1-img2,1)

def corr(img1, img2):
    return scipy.stats.pearsonr(img1.flatten(),img2.flatten()).statistic

def create_circular_mask(yDim, xDim, center=None, radius=None):

    if center is None: # use the middle of the image
        center = (int(xDim/2), int(yDim/2))
    if radius is None: # use the smallest distance between the center and image walls
        radius = min(center[0], center[1], xDim-center[0], yDim-center[1])

    Y, X = np.ogrid[:yDim, :xDim]
    dist_from_center = np.sqrt((X - center[0])**2 + (Y-center[1])**2)

    mask = dist_from_center <= radius
    return mask

def apply_mask(img, mask):
    return np.multiply(img,mask)

def apply_Fourier_mask(img, mask):
    fft2d_values = np.fft.fftshift(np.fft.fft2(img))
    filtered_fft2d_values = fft2d_values * mask
    return np.real(np.fft.ifft2(np.fft.ifftshift(filtered_fft2d_values)))

def calculate_Fourier_amplitude(img):
    fft2d_values = np.fft.fftshift(np.fft.fft2(img))
    return np.abs(fft2d_values)


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
def showPair(obj0, obj1, title0, title1, style="image", xlabel="", colorscale='gray', x0=0, y0=0, dx=1, dy=1):
    if style=="image":
        z0 = min(np.min(obj0),np.min(obj1))
        z1 = max(np.max(obj0),np.max(obj1))

        fig = make_subplots(rows=1, cols=2, subplot_titles=(title0, title1))
        fig.add_trace(go.Heatmap(z=obj0, zmin=z0, zmax=z1, colorscale=colorscale, x0=x0, y0=y0, dx=dx, dy=dy, showscale=False), row=1, col=1)
        fig.add_trace(go.Heatmap(z=obj1, zmin=z0, zmax=z1, colorscale=colorscale, x0=x0, y0=y0, dx=dx, dy=dy), row=1, col=2)
        fig.update_xaxes(scaleanchor='y1', constrain='domain', row=1, col=1)
        fig.update_xaxes(scaleanchor='y2', constrain='domain', row=1, col=2)
        #fig.update_layout(xaxis=dict(scaleanchor='y',constrain='domain'))
    elif style=="plot":
        df = pd.DataFrame()
        df[title0]=obj0[1]
        df[title1]=obj1[1]
        df[xlabel]=obj0[0]
        fig = px.line(df, x=xlabel,y=[title0, title1])
    fig.show()

In [ ]:
# Read the map
import mrcfile
with mrcfile.open(pdbID+".mrc") as mrcVol:
    vol = np.array(mrcVol.data)

# Mirror images

Given some Euler angles, $(rot, tilt, psi)$, the projection for these angles is exactly the same as the projection from $(rot, tilt+180, -(180+psi))$ except for a vertical mirror.

In [ ]:
rot0=50; tilt0=60; psi0=90;
proj=generate_projections_ASTRA(vol, [psi0,tilt0,rot0])
projMirrored=generate_projections_ASTRA(vol, [-(180+psi0),tilt0+180,rot0])
showPair(proj,projMirrored,'Proj','Proj mirrored')
showPair(proj,np.flip(projMirrored,axis=0),'Proj','Mirror(Proj mirrored)')

# Exploring the landscape of solutions: Shift

Let us see how the shift affects different similarity/loss functions.

In [ ]:
rot0=0; tilt0=0; psi0=0;
rot1=0; tilt1=5; psi1=0;

proj0=generate_projections_ASTRA(vol, [psi0, tilt0, rot0])
proj1=generate_projections_ASTRA(vol, [psi1, tilt1, rot1])

showPair(proj0,proj1,'Proj0','Proj1')


In [ ]:
def shiftExploration(maxShift, shiftStep, imageSource, imageTarget, objectiveFun, wrapping):
    # See https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.shift.html
    #     for all the wrapping possibilities
    objectiveLandscape=np.zeros((2*int(maxShift/shiftStep)+1,2*int(maxShift/shiftStep)+1))
    i=0
    for xshift in range(-maxShift,maxShift+1,shiftStep):
        j=0
        for yshift in range(-maxShift,maxShift+1,shiftStep):
            imageShifted = ndimage.shift(imageSource, [yshift, xshift], mode=wrapping)
            objectiveLandscape[i,j]=objectiveFun(imageTarget,imageShifted)
            j+=1
        i+=1
    return objectiveLandscape

maxShift=125
shiftStep=5
landscape00=shiftExploration(maxShift, shiftStep, proj0,proj0, MSE, "constant");
landscape01=shiftExploration(maxShift, shiftStep, proj0,proj1, MSE, "constant");
showPair(landscape00, landscape01, 'Proj0 vs Proj0', 'Proj0 vs Proj1', colorscale='Plasma',
         x0=-maxShift, y0=-maxShift, dx=shiftStep, dy=shiftStep)

In [ ]:
shiftRange = range(-maxShift,maxShift+1,shiftStep)
xv, yv = np.meshgrid(shiftRange, shiftRange, indexing='ij')
fig = go.Figure(data=[go.Mesh3d(x=xv.ravel(), y=yv.ravel(), z=landscape00.ravel(),
                   alphahull=5,
                   opacity=0.4,
                   intensity=landscape00.ravel(),
                   colorscale='Viridis')])
fig.show()

The wrapping method affects the calculation of the distance. For instance, if we apply the shift in Fourier space, we are implicitly making a wrapping. See https://spec.oneapi.io/oneipl/0.6/image/image-borders.html for a graphical representation of the different possibilities.

![wrappingMethods](https://drive.google.com/uc?export=view&id=1WeeJQXiyv_U5elyRlH923kb_gENH322G "blog-image align")

In [ ]:
landscape00W=shiftExploration(maxShift, shiftStep, proj0,proj0, MSE, "wrap");
landscape01W=shiftExploration(maxShift, shiftStep, proj0,proj1, MSE, "wrap");
showPair(landscape00W, landscape01W, 'Proj0W vs Proj0W', 'Proj0W vs Proj1W', colorscale='Plasma')

In [ ]:
showPair(landscape00-landscape00W, landscape01-landscape01W, 'Proj0-Proj0W', 'Proj1-Proj1W',
         colorscale='Plasma')

The landscape is also strongly affected by the distance/similarity measure.

In [ ]:
landscape00=shiftExploration(maxShift, shiftStep, proj0,proj0, corr, "constant");
landscape01=shiftExploration(maxShift, shiftStep, proj0,proj1, corr, "constant");
showPair(landscape00, landscape01, 'Proj0 vs Proj0', 'Proj0 vs Proj1', colorscale='Plasma')

# Exploring the landscape of solutions: Rotation

Let us see how the rotation affects different similarity/loss functions.

In [ ]:
def rotationExploration(maxRot, rotStep, imageSource, imageTarget, objectiveFun, wrapping):
    # See https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.shift.html
    #     for all the wrapping possibilities
    objectiveLandscape=np.zeros(2*int(maxRot/rotStep)+1)
    i=0
    for rot in range(-maxRot,maxRot+1,rotStep):
        imageRotated = ndimage.rotate(imageSource, rot, reshape=False, mode=wrapping)
        objectiveLandscape[i]=objectiveFun(imageTarget,imageRotated)
        i+=1
    return objectiveLandscape

In [ ]:
maxRot=180
rotStep=5
extent=np.arange(-maxRot,maxRot+rotStep,rotStep)
landscape00=rotationExploration(maxRot, rotStep, proj0, proj0, MSE, "constant");
landscape01=rotationExploration(maxRot, rotStep, proj0, proj1, MSE, "constant");
showPair([extent,landscape00],[extent,landscape01],'Proj0 vs Proj0','Proj0 vs Proj1','plot','Rotation')

# Exploring the landscape of solutions: Noise

Let us see how noise affects different similarity/loss functions.

$SNR=S^2/N^2$

In [ ]:
def addNoise(img, SNR):
    S2=np.mean(np.power(proj0[proj0>0],2.0))
    sigmaN=math.sqrt(S2/SNR)
    return img+np.random.normal(0,sigmaN,img.shape)

In [ ]:
SNR=0.1
noisy0 = addNoise(proj0, SNR)
noisy1 = addNoise(proj1, SNR)
showPair(noisy0, noisy1, 'Noisy0', 'Noisy1')

In [ ]:
# Let's see the effect of noise on rotational alignment
maxRot=180
rotStep=5
extent=np.arange(-maxRot,maxRot+rotStep,rotStep)
landscape00=rotationExploration(maxRot, rotStep, noisy0, proj0, MSE, "constant");
landscape01=rotationExploration(maxRot, rotStep, noisy0, proj1, MSE, "constant");
showPair([extent,landscape00],[extent,landscape01],'Noisy0 vs Proj0','Noisy0 vs Proj1','plot','Rotation')

In [ ]:
maxRot=180
rotStep=5
extent=np.arange(-maxRot,maxRot+rotStep,rotStep)
landscape00=rotationExploration(maxRot, rotStep, noisy0, proj0, corr, "constant");
landscape01=rotationExploration(maxRot, rotStep, noisy0, proj1, corr, "constant");
showPair([extent,landscape00],[extent,landscape01],'Noisy0 vs Proj0','Noisy0 vs Proj1','plot','Rotation')

On average, we should converge towards the smooth landscape

In [ ]:
SNR=0.1
df=pd.DataFrame()
yNames = []
for i in range(30):
    noisy0 = addNoise(proj0, SNR)
    landscape00=rotationExploration(maxRot, rotStep, noisy0, proj0, MSE, "constant");
    yName='Instance%03d'%i
    yNames.append(yName)
    df[yName]=landscape00
df = df.copy() # To defragment the dataframe


In [ ]:
df['Rotation']=extent
fig = px.line(df,x='Rotation',y=yNames)
fig.update_layout(showlegend=False)

In [ ]:
df['avg'] = df[yNames].mean(axis=1)
fig = px.line(df,x='Rotation',y='avg')
fig.update_layout(showlegend=False)

Still, we have peaks corresponding to -180, -90, 0, 90, and 180 degrees. The reason is that for these angles, the rotated noisy image fits exactly within the output image, so that it gets more noise.

In [ ]:
imageRotated90 = ndimage.rotate(noisy0, 90.0, reshape=False, mode="constant")
imageRotated45 = ndimage.rotate(noisy0, 45.0, reshape=False, mode="constant")
showPair(imageRotated90, imageRotated45, 'Rot=90', 'Rot=45')

For this reason, it is not the same rotating the experimental image or the reference image


In [ ]:
landscapeER=rotationExploration(maxRot, rotStep, noisy0, proj0, MSE, "constant");
landscapeRE=rotationExploration(maxRot, rotStep, proj0, noisy0, MSE, "constant");
showPair([extent, landscapeER],[extent, landscapeRE],'Rot(Exp) vs Reprojection', 'Exp vs Rot(Reprojection)','plot')

# The effect of a real-space mask

In [ ]:
mask = create_circular_mask(noisy0.shape[1], noisy0.shape[0], radius=90)
maskedNoisy0 = apply_mask(noisy0, mask)
showPair(noisy0, maskedNoisy0, "Noisy", "Masked noisy radius=90")

In [ ]:
df=pd.DataFrame()
yNames=[]
for radius in np.arange(80,125*math.sqrt(2.0),10):
    mask = create_circular_mask(noisy0.shape[1], noisy0.shape[0], radius=radius)
    maskedNoisy0 = apply_mask(noisy0, mask)
    landscapeRE=rotationExploration(maxRot, rotStep, proj0, maskedNoisy0, MSE, "constant");
    yName="Radius %d"%int(radius)
    df[yName]=landscapeRE
    yNames.append(yName)
df['Rotation']=extent
fig = px.line(df,x='Rotation',y=yNames)
fig.show()

In [ ]:
def showLandscape(df, yAxis, yNames, xName, xAxis):
    Ydim=len(yNames)
    Xdim=len(df[xAxis])
    landscape=np.zeros((Ydim,Xdim))
    for y in range(Ydim):
        landscape[y,:]=df[yNames[y]]

    f=scipy.interpolate.RegularGridInterpolator((yAxis, df[xAxis]), landscape, bounds_error=False, fill_value=None)
    newY=np.linspace(np.min(yAxis),np.max(yAxis),100)
    newX=np.linspace(np.min(df[xAxis]),np.max(df[xAxis]),100)
    X,Y = np.meshgrid(newY,newX)
    newLandscape=f((X,Y))

    fig=px.imshow(newLandscape, labels=dict(x=xName, y=xAxis, color="Landscape"),
                  x=newY, y=newX)
    fig.show()
    return newLandscape
landscape = showLandscape(df, np.arange(80,125*math.sqrt(2.0),10), yNames, "Radius", "Rotation")

It is not the same comparing a masked image and a reference image to evaluating the objective function within a mask. For instance, for the correlation.

In [ ]:
def corr(img1, img2):
    return scipy.stats.pearsonr(img1.flatten(),img2.flatten()).statistic

mask = create_circular_mask(noisy0.shape[1], noisy0.shape[0], radius=90)
maskedNoisy0 = apply_mask(noisy0, mask)

def corrWithinMask(img1, img2):
    return scipy.stats.pearsonr(img1[mask>0],img2[mask>0]).statistic

df=pd.DataFrame()
df["No mask"]=rotationExploration(maxRot, rotStep, proj0, noisy0, corr, "wrap");
df["Masked experimental"]=rotationExploration(maxRot, rotStep, proj0, maskedNoisy0, corr, "wrap");
df["Evaluation within mask"]=rotationExploration(maxRot, rotStep, proj0, noisy0, corrWithinMask, "wrap");
df['Rotation']=extent
fig = px.line(df,x='Rotation',y=["No mask", "Masked experimental", "Evaluation within mask"])
fig.show()


They are not all the same even if we normalize the maximum (although no mask and masked experimental are similar). They decay differently. The evaluation within the mask is more sensitive (its slope at the maximum is larger).

In [ ]:
df["No mask"]/=np.max(df["No mask"])
df["Masked experimental"]/=np.max(df["Masked experimental"])
df["Evaluation within mask"]/=np.max(df["Evaluation within mask"])
df['Rotation']=extent
fig = px.line(df,x='Rotation',y=["No mask", "Masked experimental", "Evaluation within mask"])
fig.show()

# Effect of a mask in Fourier space

In [ ]:
# Let us see the amplitude of the original image
A0 = 2*np.log10(1+calculate_Fourier_amplitude(proj0))
showPair(proj0/np.max(proj0), A0/np.max(A0), "Proj0", "log10(Proj0) amplitude spectrum")


In [ ]:
# Let us now filter the image, and show its spectrum
mask = create_circular_mask(proj0.shape[0],proj0.shape[1],radius=10) # 10 Fourier pixels
filteredProj0 = apply_Fourier_mask(proj0, mask)

filteredA0 = 2*np.log10(1+calculate_Fourier_amplitude(filteredProj0))
showPair(filteredProj0/np.max(filteredProj0), filteredA0/np.max(filteredA0),
         "Filtered Proj0", "log10(FilteredProj0) amplitude spectrum")


Given a Fourier index, $k$, of a cubic volume of size, $N\times N\times N$, the corresponding continuous frequency, in A$^{-1}$, is $f_k=\frac{k}{NT_s}$ where $T_s$ is the pixel size, in A. Rather than frequency, we prefer talking about resolution. $R_k=\frac{NT_s}{k}$. Remember that, for a centered Fourier transform, $k=-N/2, ..., N/2-1$. For instance, for $N=250$, then

In [ ]:
N=250
Ts=1; # A
k=np.arange(1, N/2)
px.line(k,N*Ts/k).update_layout(xaxis_title='k', yaxis_title='Rk')

In [ ]:
showPair(apply_Fourier_mask(proj0, create_circular_mask(250,250,radius=10)),
         apply_Fourier_mask(proj0, create_circular_mask(250,250,radius=30)),
         "Radius=10", "Radius=30")
showPair(apply_Fourier_mask(proj0, create_circular_mask(250,250,radius=50)),
         apply_Fourier_mask(proj0, create_circular_mask(250,250,radius=70)),
         "Radius=50", "Radius=70")
showPair(apply_Fourier_mask(proj0, create_circular_mask(250,250,radius=90)),
         apply_Fourier_mask(proj0, create_circular_mask(250,250,radius=110)),
         "Radius=90", "Radius=110")

In [ ]:
# Let us see the same for the noisy projection
showPair(apply_Fourier_mask(noisy0, create_circular_mask(250,250,radius=10)),
         apply_Fourier_mask(noisy0, create_circular_mask(250,250,radius=30)),
         "Radius=10", "Radius=30")
showPair(apply_Fourier_mask(noisy0, create_circular_mask(250,250,radius=50)),
         apply_Fourier_mask(noisy0, create_circular_mask(250,250,radius=70)),
         "Radius=50", "Radius=70")
showPair(apply_Fourier_mask(noisy0, create_circular_mask(250,250,radius=90)),
         apply_Fourier_mask(noisy0, create_circular_mask(250,250,radius=110)),
         "Radius=90", "Radius=110")

In [ ]:
# Let us explore now the effect of different Fourier filters on the evaluation of the similarity
df=pd.DataFrame()
yNames=[]
for radius in np.arange(10,110,10):
    mask = create_circular_mask(noisy0.shape[1], noisy0.shape[0], radius=radius)
    maskedNoisy0 = apply_Fourier_mask(noisy0, mask)
    landscapeRE=rotationExploration(maxRot, rotStep, proj0, maskedNoisy0, MSE, "constant");
    yName="Fourier radius %d"%int(radius)
    df[yName]=landscapeRE
    yNames.append(yName)
df['Rotation']=extent
fig = px.line(df,x='Rotation',y=yNames)
fig.show()

In [ ]:
landscape = showLandscape(df, np.arange(10,110,10), yNames, "Fourier radius", "Rotation")

# How discriminative we can be at a given frequency

In [ ]:
# With only 10 Fourier pixels
maskedNoisy0 = apply_mask(noisy0, create_circular_mask(noisy0.shape[1], noisy0.shape[0], radius=90))
filteredMaskedNoisy0 = apply_Fourier_mask(maskedNoisy0, create_circular_mask(noisy0.shape[1], noisy0.shape[0],
                                                                             radius=10))

maxRot=180
rotStep=5
extent=np.arange(-maxRot,maxRot+rotStep,rotStep)
landscapeRE0=rotationExploration(maxRot, rotStep, proj0, filteredMaskedNoisy0, corr, "constant");
landscapeRE1=rotationExploration(maxRot, rotStep, proj1, filteredMaskedNoisy0, corr, "constant");
showPair([extent,landscapeRE0], [extent,landscapeRE1], "Exp vs Proj0", "Exp vs Proj1", "plot")

We would get the wrong direction, as Proj1 correlation is higher than Proj0

In [ ]:
# With 70 Fourier pixels
maskedNoisy0 = apply_mask(noisy0, create_circular_mask(noisy0.shape[1], noisy0.shape[0], radius=90))
filteredMaskedNoisy0 = apply_Fourier_mask(maskedNoisy0, create_circular_mask(noisy0.shape[1], noisy0.shape[0],
                                                                             radius=50))

maxRot=180
rotStep=5
extent=np.arange(-maxRot,maxRot+rotStep,rotStep)
landscapeRE0=rotationExploration(maxRot, rotStep, proj0, filteredMaskedNoisy0, corr, "constant");
landscapeRE1=rotationExploration(maxRot, rotStep, proj1, filteredMaskedNoisy0, corr, "constant");
showPair([extent,landscapeRE0], [extent,landscapeRE1], "Exp vs Proj0", "Exp vs Proj1", "plot")

Now we get the right answer

# There is an interplay between shift and rot

In [ ]:
noisy0Shifted = ndimage.shift(noisy0, [10,-10], mode='wrap')
maskedNoisy0 = apply_mask(noisy0Shifted, create_circular_mask(noisy0.shape[1], noisy0.shape[0], radius=90))
filteredMaskedNoisyShifted0 = apply_Fourier_mask(maskedNoisy0, create_circular_mask(noisy0.shape[1], noisy0.shape[0],
                                                                                    radius=50))

showPair(noisy0Shifted, filteredMaskedNoisyShifted0, "Shifted noisy0", "Filtered, masked, shifted noisy0")
landscapeRE0=rotationExploration(maxRot, rotStep, proj0, filteredMaskedNoisyShifted0, corr, "constant");
landscapeRE1=rotationExploration(maxRot, rotStep, proj1, filteredMaskedNoisyShifted0, corr, "constant");
showPair([extent,landscapeRE0], [extent,landscapeRE1], "Shifted Exp vs Proj0", "Shifted Exp vs Proj1", "plot")

If the experimental image is shifted, now we get the wrong reference and the wrong rotation. This highlights the need for making a fully 3D search.

# We may combine multiple similarities

In [ ]:
# Let us explore now the effect of different Fourier filters on the evaluation of the similarity
df=pd.DataFrame()
yNames=[]
for radius in np.arange(10,110,10):
    mask = create_circular_mask(noisy0.shape[1], noisy0.shape[0], radius=radius)
    maskedNoisy0 = apply_Fourier_mask(noisy0, mask)
    landscapeRE=rotationExploration(maxRot, rotStep, proj0, maskedNoisy0, corr, "constant");
    landscapeRE=(landscapeRE-np.min(landscapeRE))/(np.max(landscapeRE)-np.min(landscapeRE))
    yName="Fourier radius %d"%int(radius)
    df[yName]=landscapeRE
    yNames.append(yName)
df['Rotation']=extent
fig = px.line(df,x='Rotation',y=yNames)
fig.show()

In [ ]:
df['product'] = df[yNames].prod(axis=1)
fig = px.line(df,x='Rotation',y='product')
fig.update_layout(showlegend=False)

# Best shift and rotation between two images

We may use the cross-correlation property of Fourier space to find the best shift and best rotation between two images.

In [ ]:
def compute_shift(img1, img2):
    """Compute the x and y shifts between two images."""

    # Compute Fourier transform of both images
    f1 = np.fft.fft2(img1)
    f2 = np.fft.fft2(img2)

    # Compute the cross-power spectrum
    cross_power = f1 * f2.conj()
    cross_power /= np.abs(cross_power)

    # Compute cross-correlation
    cross_corr = np.abs(np.fft.ifft2(cross_power))

    # Find the position of the peak
    y_shift, x_shift = np.unravel_index(np.argmax(cross_corr), cross_corr.shape)

    # Adjust shifts if they are greater than half the image dimensions
    if x_shift > img1.shape[1] // 2:
        x_shift -= img1.shape[1]
    if y_shift > img1.shape[0] // 2:
        y_shift -= img1.shape[0]

    return y_shift, x_shift

# Let us check that they work
compute_shift(proj0, ndimage.shift(proj0, [-10,20], mode='wrap'))



In [ ]:
from skimage.transform import warp_polar

def compute_rotation(img1, img2):
    """Compute the rotation between two images."""

    # Convert images to polar coordinates
    radius = img1.shape[0]/2-1
    polar_img1 = warp_polar(img1, radius=radius)
    polar_img2 = warp_polar(img2, radius=radius)

    # Compute the shift in the polar domain
    rotation, _ = compute_shift(polar_img1, polar_img2)

    return -rotation

compute_rotation(proj0, ndimage.rotate(proj0, -15, reshape=False, mode='wrap'))

# Global angular search

Given an arbitrary experimental projection, let us look for the image from a gallery that best matches the experimental one.

In [ ]:
# Let us construct the gallery of projections
def fibonacci_sphere(samples=1000):
    """Generate a set of points on the surface of a unit sphere using the Fibonacci lattice method."""
    phi = np.pi * (3. - np.sqrt(5.))  # golden angle in radians
    y = 1 - (np.arange(samples) / float(samples - 1)) * 2  # y goes from 1 to -1
    radius = np.sqrt(1 - y * y)  # radius at y position

    theta = phi * np.arange(samples)  # golden angle increment

    x, z = np.cos(theta) * radius, np.sin(theta) * radius
    return list(zip(x, y, z))

Npoints = 100
points = fibonacci_sphere(Npoints)
df = pd.DataFrame(points, columns=['x', 'y', 'z'])


# We convert the (x,y,z) points into a list of rot and tilt angles
tilt = np.rad2deg(np.arccos(df['z']))
rot = np.rad2deg(np.arctan2(df['y'],df['x']))

# Create projections
refAngles = [(0,tilti,roti) for tilti,roti in zip(tilt,rot)]
projSize=vol.shape[0]
referenceProjections = np.zeros((Npoints, projSize, projSize))
for i in range(Npoints):
    referenceProjections[i,:,:]=generate_projections_ASTRA(vol,refAngles[i])
px.scatter_3d(df,x='x',y='y', z='z')

In [ ]:
showPair(referenceProjections[0,:,:],referenceProjections[50,:,:],'Proj0','Proj50')

In [ ]:
# Let us create the experimental image
chooseRef = True
truePsi = 0
trueDeltay = 15
trueDeltax = -10
if chooseRef:
    cleanProj = referenceProjections[34,:,:]
    cleanProj = ndimage.rotate(cleanProj, truePsi, reshape=False, mode='wrap')
else:
    cleanProj = generate_projections_ASTRA(vol,(truePsi,90,100))

noisy0 = addNoise(cleanProj, SNR=100)
noisy0 = ndimage.shift(noisy0, [trueDeltay,trueDeltax], mode='wrap')
px.imshow(noisy0, color_continuous_scale='gray')

In [ ]:
# Global search TR
Nref = len(refAngles)
refCorr = np.zeros(Nref)
in_plane = np.zeros((Nref, 3))
reorientedExps = np.zeros((Nref,projSize,projSize))
for i in range(Nref):
    refProj = referenceProjections[i,:,:]

    # Look for best shift and correct
    [deltay,deltax] = compute_shift(refProj, noisy0)
    reorientedExps[i,:,:] = ndimage.shift(noisy0, [deltay,deltax], mode='wrap')

    # Look for best rotation and correct
    psi = compute_rotation(refProj, reorientedExps[i,:,:])
    reorientedExps[i,:,:] = ndimage.rotate(reorientedExps[i,:,:], psi, reshape=False, mode='wrap')

    # Now compare the reoriented experimental image with the reference image
    refCorr[i]=corr(refProj, reorientedExps[i,:,:])
    in_plane[i,:]=[psi, deltay, deltax]


In [ ]:
px.line(y=refCorr)

In [ ]:
iBestRef=np.argmax(refCorr)
print("Best reference is %d"%iBestRef)
print("   In-plane parameters: psi=%f deltay=%f deltax=%f"%
      (in_plane[iBestRef,0],in_plane[iBestRef,1],in_plane[iBestRef,2]))
print("   Tilt=%f Rot=%f"%(refAngles[i][1],refAngles[i][2]))


showPair(referenceProjections[iBestRef,:,:],reorientedExps[iBestRef,:,:],
         'Reference','Reoriented Exp')

In [ ]:
# Global search RT
Nref = len(refAngles)
refCorr = np.zeros(Nref)
in_plane = np.zeros((Nref, 3))
reorientedExps = np.zeros((Nref,projSize,projSize))
for i in range(Nref):
    refProj = referenceProjections[i,:,:]

    # Look for best rotation and correct
    psi = compute_rotation(refProj, noisy0)
    reorientedExps[i,:,:] = ndimage.rotate(noisy0, psi, reshape=False, mode='wrap')

    # Look for best shift and correct
    [deltay,deltax] = compute_shift(refProj, reorientedExps[i,:,:])
    reorientedExps[i,:,:] = ndimage.shift(reorientedExps[i,:,:], [deltay,deltax], mode='wrap')

    # Now compare the reoriented experimental image with the reference image
    refCorr[i]=corr(refProj, reorientedExps[i,:,:])
    in_plane[i,:]=[psi, deltay, deltax]


In [ ]:
px.line(y=refCorr)

In [ ]:
iBestRef=np.argmax(refCorr)
print("Best reference is %d"%iBestRef)
print("   In-plane parameters: psi=%f deltay=%f deltax=%f"%
      (in_plane[iBestRef,0],in_plane[iBestRef,1],in_plane[iBestRef,2]))
print("   Tilt=%f Rot=%f"%(refAngles[i][1],refAngles[i][2]))


showPair(referenceProjections[iBestRef,:,:],reorientedExps[iBestRef,:,:],
         'Reference','Reoriented Exp')

### In summary ...


1.   The orientation of a particle is given by 5 parameters: rot and tilt define the projection direction. Then, we have 3 parameters for in-plane transformations: 1 rotation and 2 shifts (x,y)
2.   In SPA, the projection of the object of interest should always lie within a spherical mask whose maximum radius is half the size of the particle size.
3. We can save half of the reference projections if we consider mirrors.
4. When comparing two images, we must define a metric for distance or similarity. This metric, contingent upon the five parameters governing alignment, outlines a 'landscape of solutions'—essentially a multidimensional space where each point represents the 'fitness' or 'quality' of a potential solution in terms of how well it aligns the two images.
5. We have to compare rotated and translated versions of I2 to I1. The boundary convention adopted to generate the interpolated image affects the landscape of solutions.
6. For the same reason, the distance is not symmetric. That is, it is not the same $d(I_1,T(I_2))$ as $d(T^{-1}(I_1),I_2)$.
7. Noise makes the landscape much more ruggy and the global maximum of the noisy landscape may not coincide with the true global maximum. In the field, we normally refer to local maxima, but these local maxima may actually refer to genuine local maxima or to "shifted" global maxima.
8. The landscape is affected by masking in real and Fourier space.
9. The way to use the mask also affects.
10. The metrics can be combined into a more robust metric.
11. There are fast ways to look for the shift or the rotation exploiting the correlation theorem of the Fourier transform.
12. However, the search for the shift using the Fourier transform assumes that the rotation is correct and viceversa. For this reason, it is not the same looking for the shift and then for the rotation or the oposite. And none of the two strategies is necessarily better than the other. It depends on the specific pair of images and their landscape of solutions.
13. A global search can be done fully in 5D (an exhaustive search over all parameter combinations) or it can be accelerated in a 2D+1D+2D search (rot and tilt, in-plane rotation, in-plane shift). This second approach is much faster, but it may lead to suboptimal results, specially in the presence of high levels of noise.
